# Privacy

Do not commit notebook outputs. Local runs may print or display record-level
research text, Reddit IDs, or developer paths. Keep outputs cleared in git.
Public examples live in `tests/fixtures/synthetic/` (fully synthetic).
See `docs/data_statement.md`.


In [ ]:
# ============================================
# Reddit AI-bias summary report/visualization (Notebook)
# Prerequisites:
#   data/filtered/_summaries/overview_by_subreddit.csv
#   data/filtered/_summaries/category_breakdown.csv
# (if exists)
#   data/filtered/_summaries/top_keywords.csv
#   data/filtered/_summaries/examples_per_category.csv
# ============================================
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from textwrap import shorten

# ---------- 0) Paths ----------
BASE = Path("data/filtered/_summaries")
OV_PATH = BASE / "overview_by_subreddit.csv"
CB_PATH = BASE / "category_breakdown.csv"
TK_PATH = BASE / "top_keywords.csv"              
EX_PATH = BASE / "examples_per_category.csv"    

assert OV_PATH.exists(), f"Missing {OV_PATH}"
assert CB_PATH.exists(), f"Missing {CB_PATH}"

# ---------- 1) Loading ----------
df_overview = pd.read_csv(OV_PATH)  # columns: subreddit, rows
df_cat = pd.read_csv(CB_PATH)       # columns: subreddit, bias_type, cnt

# (Option) 있으면 load
df_kw = pd.read_csv(TK_PATH) if TK_PATH.exists() else None
df_ex = pd.read_csv(EX_PATH) if EX_PATH.exists() else None

# ---------- 2) Basic metrics ----------
df_overview = df_overview.sort_values("rows", ascending=False).reset_index(drop=True)
total_rows = int(df_overview["rows"].sum())

print("=== Overview ===")
display(df_overview.assign(share_pct=lambda d: (d["rows"]/total_rows*100).round(2)))

print(f"\nTotal rows = {total_rows:,}")

# ---------- 3) (subreddit × category) ----------
pivot_cnt = df_cat.pivot_table(index="subreddit", columns="bias_type", values="cnt", aggfunc="sum", fill_value=0)
pivot_cnt = pivot_cnt.loc[df_overview["subreddit"]]  

# Percent (row normalization)
pivot_pct = pivot_cnt.div(pivot_cnt.sum(axis=1), axis=0).replace([np.inf, np.nan], 0) * 100

print("\n=== Counts pivot (top-left) ===")
display(pivot_cnt.iloc[:5, :10])

print("\n=== Percent pivot (top-left) ===")
display(pivot_pct.iloc[:5, :10].round(1))

# ---------- 4) Concentration (HHI) by category ----------
# HHI = sum_i (share_i^2). 0~1 (higher = more concentrated)
hhi = (pivot_pct/100).pow(2).sum(axis=1).rename("hhi")
df_overview = df_overview.merge(hhi, left_on="subreddit", right_index=True, how="left")
print("\n=== Concentration (HHI) by subreddit ===")
display(df_overview.sort_values(["hhi", "rows"], ascending=[False, False]).reset_index(drop=True))

# ---------- 5) Subreddit-wise top-k categories ----------
def topk_categories(pct_table, k=5):
    out = []
    for sub, row in pct_table.iterrows():
        top = row.sort_values(ascending=False).head(k)
        out.append(pd.DataFrame({
            "subreddit": sub,
            "bias_type": top.index,
            "share_pct": top.values.round(1)
        }))
    return pd.concat(out, ignore_index=True)

topk_cat = topk_categories(pivot_pct, k=5)
print("\n=== Top-5 categories per subreddit (share %) ===")
display(topk_cat)

# ---------- 6) Visualization ----------
plt.figure(figsize=(10, 5))
plt.bar(df_overview["subreddit"], df_overview["rows"])
plt.title("Rows per Subreddit")
plt.xticks(rotation=45, ha="right")
plt.ylabel("rows")
plt.tight_layout()
plt.show()

# Normalized stacked horizontal bar (top 6 subreddits)
keep_subs = df_overview["subreddit"].head(6).tolist()
stack_data = pivot_pct.loc[keep_subs]
cats = stack_data.columns.tolist()

fig, ax = plt.subplots(figsize=(10, 6))
left = np.zeros(len(stack_data))
for c in cats:
    vals = stack_data[c].values
    ax.barh(stack_data.index, vals, left=left, label=c)
    left += vals
ax.set_title("Bias-type Share by Subreddit (Top 6)")
ax.set_xlabel("share %")
ax.legend(bbox_to_anchor=(1.02,1), loc="upper left", frameon=False, ncol=1, fontsize=9)
plt.tight_layout()
plt.show()

# ---------- 7) Drill-down function ----------
def show_subreddit_detail(subreddit: str, topk_keywords: int = 20, sample_per_cat: int = 3):
    print(f"\n### Subreddit: {subreddit}")
    # Scale
    row_cnt = int(df_overview.loc[df_overview["subreddit"] == subreddit, "rows"].sum())
    hhi_val = float(df_overview.loc[df_overview["subreddit"] == subreddit, "hhi"].fillna(0).iloc[0])
    print(f"- rows: {row_cnt:,}  |  HHI: {hhi_val:.3f}")

    # Categories distribution (top 10)
    if subreddit in pivot_cnt.index:
        dist = pivot_cnt.loc[subreddit].sort_values(ascending=False)
        share = pivot_pct.loc[subreddit].reindex(dist.index)
        tmp = pd.DataFrame({"cnt": dist, "share_pct": share.round(1)}).head(10)
        print("\n- Top categories (by count):")
        display(tmp)
    else:
        print("\n- No category info found.")

    # Keywords (if exists)
    if df_kw is not None and "keyword" in df_kw.columns:
        kw_sub = df_kw[df_kw["subreddit"] == subreddit].copy()
        if not kw_sub.empty:
            print(f"\n- Top keywords (top {topk_keywords}):")
            display(kw_sub.sort_values("cnt", ascending=False).head(topk_keywords))
        else:
            print("\n- No keywords found for this subreddit.")
    else:
        print("\n- top_keywords.csv not available.")

    # Examples (if exists)
    if df_ex is not None and set(["subreddit","bias_type","id","title","selftext"]).issubset(df_ex.columns):
        ex_sub = df_ex[df_ex["subreddit"] == subreddit].copy()
        if not ex_sub.empty:
            print("\n- Examples (one line per sample):")
            # Category-wise samples
            for bt, grp in ex_sub.groupby("bias_type"):
                print(f"\n  [{bt}]")
                g = grp.head(sample_per_cat)
                for _, r in g.iterrows():
                    text = (str(r.get("title","")) + " " + str(r.get("selftext",""))).strip()
                    print("   -", shorten(text.replace("\n"," "), width=140, placeholder="…"))
        else:
            print("\n- No examples for this subreddit.")
    else:
        print("\n- examples_per_category.csv not available or missing columns.")

# Examples:
show_subreddit_detail("MachineLearning")
show_subreddit_detail("ArtistHate")

# ---------- 8) Compare subreddits by specific categories (e.g. gender, occupation) ----------
focus_cats = ["gender", "occupation", "race", "age"]
comp = pivot_pct[focus_cats].copy().loc[df_overview["subreddit"]]
comp = comp.fillna(0).round(1)

print("\n=== Cross-subreddit comparison for selected categories (share %) ===")
display(comp)

# Bar chart (gender share comparison)
plt.figure(figsize=(10,5))
plt.bar(comp.index, comp["gender"])
plt.title("Gender share % by subreddit")
plt.xticks(rotation=45, ha="right")
plt.ylabel("share %")
plt.tight_layout()
plt.show()
